# 00 · AI는 무엇을 알고, 무엇을 실행할까? · 35분

## 학습 목표
- 한빛테크 도쿄 출장 시나리오에서 LLM의 답변과 확인된 사실을 구분합니다.
- Codespaces 또는 로컬 VS Code의 `.venv` 커널에서 첫 실제 모델 응답을 얻습니다.
- 입력 → 모델 → 출력과 토큰 사용량을 관찰합니다. Python 문법 암기가 목표는 아닙니다.

## 개념: 먼저 역할을 나누기
| 요소 | 하는 일 | 하지 않는 일 |
|---|---|---|
| LLM | 입력으로 주어진 내용을 바탕으로 답변/도구 요청 생성 | 사내 문서나 현재 날씨를 자동으로 확인하지 않음 |
| Python 함수 | 코드가 허용한 실제 API 호출/계산 | 스스로 어떤 도구를 쓸지 판단하지 않음 |
| Agent loop | 모델 요청 → 함수 실행 → 결과 반환을 종료 조건까지 반복 | 무제한 실행이나 예약 권한을 뜻하지 않음 |
| Search | 관련 문서 조각을 찾아 반환 | 그 자체로 최종 답변을 생성하지 않음 |
| RAG | 검색 근거를 모델 입력에 넣어 답변 생성 | 모델 재학습이나 정확성 보장이 아님 |

**Foundry**는 모델, **AI Search**는 검색, **APIM**은 개인 키별 접근 제한을 제공합니다.
**Agent Framework**의 반복 제어는 지금 실행 중인 Python 환경(Codespaces/PC)에서 동작합니다.
Foundry Agent Service에 호스팅하지 않으며 학생 Azure 로그인·리소스 생성은 필요 없습니다.

### 준비와 실행 범위
1. [수강생 안내](../docs/student.md)의 Codespaces 시작 경로를 먼저 따릅니다.
2. 터미널에서 `uv sync --frozen` 실행.
3. 강사가 준 개인 파일만 저장소 루트 `.env`로 저장한 뒤 `uv run python scripts/workshop.py doctor` 실행. 키·파일 내용을 출력하지 않습니다.
4. **Select Kernel → Python Environments → .venv** 선택, 셀을 위에서 아래로 실행.
5. `doctor` 기본 점검은 오프라인입니다. `doctor --live`는 선택이며 실제 API 사용/과금이 가능합니다.

아래는 실제 Azure 모델 호출입니다. 예측/기록 셀은 로컬이고, `responses.create`부터 네트워크 요청입니다.


In [ ]:
from pathlib import Path
import sys
import json
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
from hanbit_assistant.config import Settings
from hanbit_assistant.clients import create_openai_client
settings = Settings.from_env()
# 비밀 키, 환경변수 전체, HTTP 헤더를 출력하지 않습니다.
print({"model": settings.model, "my_index": settings.search_index})


## 실험과 연습 · 예측 → 실행 → 관찰 → 변경
**예측(2분):** 문서와 도구 없이 “한빛테크 도쿄 숙박 한도”를 물으면 모델이 어떤 근거로 답할까요?
아래 `prediction`을 먼저 적고, 질문을 실행한 뒤 답에 문서 출처가 있는지 확인하세요.

**실행:** `QUESTION`은 고정한 채 기본 지침으로 한 번 호출합니다. 출력 토큰은 답변 길이 제한이며 지식의 양이 아닙니다.
**관찰:** 그럴듯한 금액/말투와 검증 가능한 근거를 구분하고 사용량을 기록합니다.
**변경:** `INSTRUCTIONS`만 “제공된 문서가 없으면 회사 규정을 확인할 수 없다고 답하라.”로 바꾸고 호출 셀만 다시 실행합니다.
두 답이 같아도 실패가 아닙니다. “문서를 보지 않았다”는 조건은 여전히 같습니다.

`settings.model`은 강사가 실제 배포한 이름이고 키는 본인 APIM 키입니다. `store=False`는 서버 측 응답 저장을 끕니다.


In [ ]:
# 연습: 빈 기록만 먼저 채우세요. 기본 질문은 그대로 실행 가능합니다.
prediction = ""  # 예: 정확한 숫자를 말해도 믿을 수 있을까?
QUESTION = "한빛테크 도쿄 출장 시 1인 1박 숙박비 한도와 세금 포함 여부를 알려줘."
INSTRUCTIONS = "한국어로 짧게 답하라."
async with create_openai_client(settings) as client:
    response = await client.responses.create(
        model=settings.model, input=QUESTION, instructions=INSTRUCTIONS,
        store=False, max_output_tokens=350,
    )
    print(response.output_text)
    print("실제 사용량:", response.usage.model_dump() if response.usage else "서비스 미제공")
assert response.output_text


## 관찰 기록과 체크포인트
- [ ] 첫 응답을 얻었다. 실패한 경우 오류 단계만 기록했다.
- [ ] 모델 입력에 회사 문서도 날씨 API 결과도 없었음을 설명했다.
- [ ] 지침 변경 전/후 답의 차이 또는 차이 없음을 적었다.

| 실행 | 바꾼 것 | 핵심 답변 | 검증 가능한 출처? |
|---|---|---|---|
| A | 기본 지침 | 직접 기록 | 직접 기록 |
| B | 지침만 변경 | 직접 기록 | 직접 기록 |

**결과 질문:** 정답처럼 보이는 답을 얻는 것과 정답의 근거를 확보하는 것은 왜 다른가요?
04에서 동일 질문을 문서와 함께 다시 사용하므로 질문과 관찰을 남겨 두세요.

<details><summary>참고 풀이 · 예시 해설 (직접 비교한 뒤 열기)</summary>
문서를 전달하지 않았으므로 숫자가 맞아도 사규를 확인한 것은 아닙니다. 답변 보류 지침은 추측을 줄이는 데 도움을 줄 수 있지만 실제 문서 접근 권한을 추가하지 않습니다. 검색과 RAG가 필요한 이유입니다.
</details>

### 실패하면
- import 실패: `.venv` 커널 선택 후 첫 셀부터 재실행.
- 401/403: 개인 파일·키/인덱스 배정은 강사와 비공개로 확인. 공유 `/openai` 경로로 바꾸지 않습니다.
- 429: 반복 클릭 중단, Retry-After가 있으면 그 안내를 따르고 강사에게 알립니다.
- 빈 출력: 실패를 기록하고 짧은 질문으로 한 번 재시도. 가짜 응답을 붙여 넣지 않습니다.
